In [ ]:
import torch
from datasets import load_dataset

import os
import json
from tqdm.auto import tqdm

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Getting Datasets for Minimind 0


https://www.modelscope.cn/datasets/gongjy/minimind_dataset/files
*   ./pretrain_hq.jsonl
*   ./sft_mini_512.jsonl



In [ ]:
# Load pretrain (Schema: {'text': ...})
pretrain_ds = load_dataset("jingyaogong/minimind_dataset",
                           data_files="pretrain_hq.jsonl",
                           split="train")

# Load SFT (Schema: {'conversations': ...})
sft_ds = load_dataset("jingyaogong/minimind_dataset",
                      data_files="sft_mini_512.jsonl",
                      split="train")

print(pretrain_ds[0])
print(sft_ds[0])

### Getting english datasets


*   roneneldan/TinyStories for Pre-training
*   HuggingFaceTB/smollm-corpus for SFT





In [ ]:
# Pre-training Data (TinyStories)
print("Loading TinyStories for Pre-training...")
# Using streaming=True to avoid downloading the entire 50GB dataset
pretrain_en = load_dataset("roneneldan/TinyStories", split="train", streaming=True).take(100000)

output_pretrain = "/data/pretrain_en.jsonl"
with open(output_pretrain, "w", encoding="utf-8") as f:
    for item in tqdm(pretrain_en, total=100000, desc="Processing Pretrain"):
        # Format: {'text': 'story content<|im_end|>'}
        new_item = {"text": f"{item['text']}<|im_end|>"}
        f.write(json.dumps(new_item, ensure_ascii=False) + "\n")

# 2. SFT Data (SmolLM-Corpus Instruct)
print("\nLoading SmolLM-Corpus for SFT...")

In [ ]:
# Using cosmopedia-v2 subset as a high-quality instruction source
sft_en = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train", streaming=True).take(20000)

output_sft = "/data/sft_en.jsonl"
with open(output_sft, "w", encoding="utf-8") as f:
    for item in tqdm(sft_en, total=20000, desc="Processing SFT"):
        # Format: {'conversations': [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}]}
        new_item = {
            "conversations": [
                {"role": "user", "content": f"Tell me about {item['prompt'] if 'prompt' in item else 'this topic'}."},
                {"role": "assistant", "content": item["text"]}
            ]
        }
        f.write(json.dumps(new_item, ensure_ascii=False) + "\n")

print(f"\nSuccess! Files are ready in the data/ folder:\n- {output_pretrain}\n- {output_sft}")